# Visual Experiment Report - CNN Kustom 6 Kelas Tanpa Tungro

Notebook ini adalah versi visual report untuk **paper 6 kelas tanpa Tungro**. Fokusnya bukan lagi membandingkan CNN Kustom dengan MobileNetV2, tetapi menyajikan hasil **CNN Kustom Stable** secara runtut, bersih, dan mudah dijelaskan.

Prinsip visualisasi yang dipakai mengikuti pendekatan *storytelling with data*:

1. Setiap grafik harus menjawab satu pertanyaan.
2. Judul grafik menyampaikan pesan utama, bukan hanya nama metrik.
3. Warna digunakan untuk mengarahkan perhatian pembaca.
4. Tabel digunakan untuk detail, grafik digunakan untuk pola.
5. Setiap visual diikuti interpretasi singkat.

## 1. Setup Notebook

Pilih kernel Python dari virtual environment project:

`C:\\Users\\farid\\Downloads\\archive (5)\\rice_leaf_disease_project\\.venv\\Scripts\\python.exe`

In [ ]:
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None

cwd = Path.cwd().resolve()
if cwd.name == "visual_reports":
    PROJECT_ROOT = cwd.parents[1]
elif cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "rice_leaf_disease_project":
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = cwd / "rice_leaf_disease_project"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing.dataset_6_classes import (
    EXCLUDED_CLASS,
    SIX_CLASS_NAMES,
    collect_image_paths_and_labels,
)

REPORT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "custom_cnn_stable_6kelas_tanpa_tungro"
SUMMARY_PATH = REPORT_DIR / f"ringkasan_{EXPERIMENT_NAME}.csv"
CLASS_REPORT_PATH = REPORT_DIR / f"classification_report_{EXPERIMENT_NAME}.txt"
TRAINING_LOG_PATH = REPORT_DIR / f"training_log_{EXPERIMENT_NAME}.csv"
TRAINING_PLOT_PATH = REPORT_DIR / f"grafik_pelatihan_{EXPERIMENT_NAME}.png"
CONFUSION_MATRIX_PATH = REPORT_DIR / f"confusion_matrix_{EXPERIMENT_NAME}.png"

BASE_COLOR = "#9AA0A6"
MAIN_COLOR = "#2F6B4F"
ACCENT_COLOR = "#C65D3A"
GRID_COLOR = "#E6E6E6"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID_COLOR,
    "grid.linewidth": 0.8,
    "font.size": 10,
})

print("Project root:", PROJECT_ROOT)
print("Experiment:", EXPERIMENT_NAME)

## 2. Status File Hasil Eksperimen

Visual report ini membutuhkan output dari notebook training `custom_cnn_6kelas_stable_experiment.ipynb`. Jika sebagian file belum ada, jalankan notebook training tersebut terlebih dahulu sampai selesai.

In [ ]:
required_files = pd.DataFrame([
    {"kebutuhan": "Ringkasan metrik", "path": SUMMARY_PATH},
    {"kebutuhan": "Classification report", "path": CLASS_REPORT_PATH},
    {"kebutuhan": "Log training", "path": TRAINING_LOG_PATH},
    {"kebutuhan": "Grafik training", "path": TRAINING_PLOT_PATH},
    {"kebutuhan": "Confusion matrix", "path": CONFUSION_MATRIX_PATH},
])

required_files["tersedia"] = required_files["path"].apply(lambda p: p.exists())
required_files["file"] = required_files["path"].apply(lambda p: p.name)
display(required_files[["kebutuhan", "file", "tersedia"]])

missing = required_files[~required_files["tersedia"]]
if len(missing) > 0:
    display(Markdown(
        "**Catatan:** beberapa file hasil belum tersedia. "
        "Jalankan notebook `custom_cnn_6kelas_stable_experiment.ipynb` sampai selesai, "
        "lalu jalankan ulang visual report ini."
    ))

## 3. Cerita Eksperimen

Pertanyaan utama visual report ini:

**Seberapa baik CNN Kustom Stable mengenali enam kondisi daun padi ketika kelas Tungro yang dominan tidak digunakan?**

Kelas Tungro dikeluarkan dari eksperimen paper karena jumlah datanya jauh lebih besar dibanding kelas lainnya. Dengan enam kelas, distribusi data menjadi lebih seimbang sehingga evaluasi model tidak terlalu dipengaruhi oleh satu kelas dominan.

## 4. Distribusi Dataset

Visual ini menjawab pertanyaan: **apakah enam kelas yang digunakan relatif seimbang?**

In [ ]:
image_paths, labels, class_names = collect_image_paths_and_labels()
distribution_df = pd.DataFrame({
    "class_name": class_names,
    "image_count": [labels.count(i) for i in range(len(class_names))],
})
distribution_df["percentage"] = distribution_df["image_count"] / distribution_df["image_count"].sum() * 100
display(distribution_df)

total_images = int(distribution_df["image_count"].sum())
min_count = int(distribution_df["image_count"].min())
max_count = int(distribution_df["image_count"].max())

display(Markdown(
    f"**Insight:** dataset paper berisi **{total_images:,} gambar** dari enam kelas. "
    f"Jumlah gambar per kelas berada pada rentang **{min_count} sampai {max_count}**, "
    "sehingga distribusinya relatif seimbang dibanding dataset 7 kelas yang memasukkan Tungro."
))

In [ ]:
plot_df = distribution_df.sort_values("image_count")

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = [MAIN_COLOR if value == plot_df["image_count"].max() else BASE_COLOR for value in plot_df["image_count"]]
bars = ax.barh(plot_df["class_name"], plot_df["image_count"], color=colors)

ax.set_title("Enam kelas memiliki jumlah gambar yang relatif seimbang", fontsize=13, fontweight="bold")
ax.set_xlabel("Jumlah gambar")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.8)
ax.grid(axis="y", visible=False)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 5, bar.get_y() + bar.get_height() / 2, f"{int(width)}", va="center")

plt.tight_layout()
plt.show()

**Interpretasi:** grafik horizontal dipakai karena nama kelas cukup panjang. Warna hijau hanya digunakan untuk menyorot kelas dengan jumlah gambar terbesar, sedangkan kelas lain dibuat abu-abu agar pembaca fokus pada pesan utama: distribusi enam kelas tidak jomplang.

## 5. Split Train, Validation, dan Test

Visual ini menjawab pertanyaan: **berapa banyak data yang digunakan untuk belajar, validasi, dan pengujian akhir?**

In [ ]:
split_counts = {
    "Train": int(round(total_images * 0.70)),
    "Validation": int(round(total_images * 0.15)),
}
split_counts["Test"] = total_images - split_counts["Train"] - split_counts["Validation"]

split_df = pd.DataFrame({
    "subset": list(split_counts.keys()),
    "image_count": list(split_counts.values()),
})
split_df["percentage"] = split_df["image_count"] / split_df["image_count"].sum() * 100
display(split_df)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(split_df["subset"], split_df["image_count"], color=[MAIN_COLOR, BASE_COLOR, ACCENT_COLOR])
ax.set_title("Sebagian besar data digunakan untuk training model", fontsize=13, fontweight="bold")
ax.set_ylabel("Jumlah gambar")
ax.grid(axis="y", alpha=0.8)
ax.grid(axis="x", visible=False)

for bar, pct in zip(bars, split_df["percentage"]):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 25, f"{int(height)}\n({pct:.1f}%)", ha="center")

plt.tight_layout()
plt.show()

**Interpretasi:** train set digunakan untuk membuat model belajar pola visual penyakit, validation set digunakan untuk memantau performa selama training, dan test set digunakan untuk evaluasi akhir. Pemisahan ini penting agar nilai test tidak berasal dari data yang sudah dilihat model saat belajar.

## 6. Ringkasan Metrik Utama

Visual ini menjawab pertanyaan: **berapa performa akhir model pada test set?**

Untuk dataset yang memiliki beberapa kelas, `F1-score macro` penting karena setiap kelas diberi bobot yang sama dalam perhitungan rata-rata.

In [ ]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
else:
    summary_df = pd.DataFrame(columns=[
        "experiment", "model", "test_accuracy", "precision_macro", "recall_macro", "f1_macro",
        "precision_weighted", "recall_weighted", "f1_weighted"
    ])

display(summary_df)

if len(summary_df) > 0:
    row = summary_df.iloc[0]
    accuracy_col = "test_accuracy" if "test_accuracy" in summary_df.columns else "accuracy"
    acc = float(row[accuracy_col])
    f1_macro = float(row["f1_macro"])
    display(Markdown(
        f"**Angka utama:** CNN Kustom Stable memperoleh **accuracy {acc * 100:.2f}%** "
        f"dan **F1-score macro {f1_macro * 100:.2f}%** pada test set."
    ))
else:
    display(Markdown(
        "**Belum ada ringkasan hasil.** Jalankan training pada notebook "
        "`custom_cnn_6kelas_stable_experiment.ipynb` sampai selesai."
    ))

In [ ]:
if len(summary_df) > 0:
    row = summary_df.iloc[0]
    accuracy_col = "test_accuracy" if "test_accuracy" in summary_df.columns else "accuracy"
    metric_df = pd.DataFrame({
        "metric": ["Accuracy", "Precision macro", "Recall macro", "F1 macro"],
        "score": [
            float(row[accuracy_col]),
            float(row["precision_macro"]),
            float(row["recall_macro"]),
            float(row["f1_macro"]),
        ],
    }).sort_values("score")

    fig, ax = plt.subplots(figsize=(8, 4))
    colors = [MAIN_COLOR if metric == "F1 macro" else BASE_COLOR for metric in metric_df["metric"]]
    ax.hlines(y=metric_df["metric"], xmin=0, xmax=metric_df["score"], color=colors, linewidth=3)
    ax.scatter(metric_df["score"], metric_df["metric"], color=colors, s=90)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Skor")
    ax.set_title("F1 macro menjadi metrik utama karena setiap kelas dihitung setara", fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.8)
    ax.grid(axis="y", visible=False)

    for score, metric in zip(metric_df["score"], metric_df["metric"]):
        ax.text(score + 0.02, metric, f"{score * 100:.2f}%", va="center")

    plt.tight_layout()
    plt.show()
else:
    display(Markdown("Grafik metrik belum dapat dibuat karena file ringkasan hasil belum tersedia."))

**Interpretasi:** dot plot dipakai karena jumlah metrik sedikit dan pembaca perlu membandingkan skor secara cepat. F1 macro diberi warna utama karena metrik ini paling relevan untuk membaca performa rata-rata antar kelas.

## 7. Proses Training Model

Visual ini menjawab pertanyaan: **apakah model belajar secara stabil dari epoch ke epoch?**

In [ ]:
if TRAINING_LOG_PATH.exists():
    history_df = pd.read_csv(TRAINING_LOG_PATH)
    display(history_df.tail())
else:
    history_df = pd.DataFrame()
    display(Markdown("Log training belum tersedia. Jalankan notebook training stable sampai selesai."))

In [ ]:
if len(history_df) > 0:
    epoch_axis = history_df["epoch"] + 1 if "epoch" in history_df.columns else np.arange(1, len(history_df) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(epoch_axis, history_df["accuracy"], label="Training", color=BASE_COLOR, linewidth=2)
    axes[0].plot(epoch_axis, history_df["val_accuracy"], label="Validation", color=MAIN_COLOR, linewidth=2)
    axes[0].set_title("Validation accuracy menunjukkan performa pada data yang tidak dilatih", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].legend()

    axes[1].plot(epoch_axis, history_df["loss"], label="Training", color=BASE_COLOR, linewidth=2)
    axes[1].plot(epoch_axis, history_df["val_loss"], label="Validation", color=ACCENT_COLOR, linewidth=2)
    axes[1].set_title("Jarak training dan validation loss membantu membaca overfitting", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    display(Markdown("Grafik training belum dapat dibuat karena log training belum tersedia."))

**Interpretasi:** grafik accuracy dan loss dibaca bersama. Jika accuracy training terus naik tetapi validation accuracy stagnan atau turun, model mulai menghafal data training. Jika validation loss turun, model biasanya masih memperbaiki kemampuan generalisasi.

## 8. Classification Report per Kelas

Visual ini menjawab pertanyaan: **kelas mana yang paling mudah dan paling sulit dikenali oleh model?**

In [ ]:
def parse_classification_report(path: Path) -> pd.DataFrame:
    rows = []
    if not path.exists():
        return pd.DataFrame(columns=["class_name", "precision", "recall", "f1_score", "support"])

    for line in path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("accuracy") or stripped.startswith("macro avg") or stripped.startswith("weighted avg"):
            continue
        match = re.match(r"^(.+?)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+(\d+)$", stripped)
        if match:
            rows.append({
                "class_name": match.group(1).strip(),
                "precision": float(match.group(2)),
                "recall": float(match.group(3)),
                "f1_score": float(match.group(4)),
                "support": int(match.group(5)),
            })
    return pd.DataFrame(rows)

class_report_df = parse_classification_report(CLASS_REPORT_PATH)
display(class_report_df)

In [ ]:
if len(class_report_df) > 0:
    f1_df = class_report_df.sort_values("f1_score")
    weakest = f1_df.iloc[0]
    strongest = f1_df.iloc[-1]

    fig, ax = plt.subplots(figsize=(9, 4.8))
    colors = [ACCENT_COLOR if cls == weakest["class_name"] else MAIN_COLOR if cls == strongest["class_name"] else BASE_COLOR for cls in f1_df["class_name"]]
    bars = ax.barh(f1_df["class_name"], f1_df["f1_score"], color=colors)
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1-score")
    ax.set_ylabel("")
    ax.set_title(f"Kelas {weakest['class_name']} menjadi kelas tersulit berdasarkan F1-score", fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.8)
    ax.grid(axis="y", visible=False)

    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.02, bar.get_y() + bar.get_height() / 2, f"{width * 100:.1f}%", va="center")

    plt.tight_layout()
    plt.show()

    display(Markdown(
        f"**Insight:** kelas dengan F1-score tertinggi adalah **{strongest['class_name']}** "
        f"({strongest['f1_score'] * 100:.1f}%), sedangkan kelas tersulit adalah "
        f"**{weakest['class_name']}** ({weakest['f1_score'] * 100:.1f}%)."
    ))
else:
    display(Markdown("Classification report belum tersedia, sehingga grafik F1-score per kelas belum dapat dibuat."))

**Interpretasi:** F1-score per kelas lebih informatif daripada accuracy umum karena menunjukkan performa model pada masing-masing penyakit. Warna oranye menandai kelas yang paling perlu dianalisis dalam pembahasan paper.

## 9. Precision vs Recall

Visual ini menjawab pertanyaan: **apakah model lebih sering salah karena prediksinya kurang tepat atau karena banyak data kelas tertentu tidak terdeteksi?**

In [ ]:
if len(class_report_df) > 0:
    pr_df = class_report_df.sort_values("recall")
    y = np.arange(len(pr_df))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(pr_df["precision"], y, color=MAIN_COLOR, s=80, label="Precision")
    ax.scatter(pr_df["recall"], y, color=ACCENT_COLOR, s=80, label="Recall")

    for idx, row in pr_df.iterrows():
        y_pos = list(pr_df.index).index(idx)
        ax.plot([row["precision"], row["recall"]], [y_pos, y_pos], color=BASE_COLOR, linewidth=2, alpha=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(pr_df["class_name"])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Skor")
    ax.set_title("Jarak precision dan recall menunjukkan jenis kesalahan model", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.8)
    ax.grid(axis="y", visible=False)
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("Grafik precision vs recall belum dapat dibuat karena classification report belum tersedia."))

**Interpretasi:** precision tinggi berarti prediksi model pada kelas tersebut cenderung tepat. Recall tinggi berarti sebagian besar gambar asli dari kelas tersebut berhasil ditemukan model. Jika recall rendah, model sering melewatkan kelas tersebut.

## 10. Confusion Matrix

Visual ini menjawab pertanyaan: **kelas mana yang sering tertukar dengan kelas lain?**

In [ ]:
if CONFUSION_MATRIX_PATH.exists():
    from IPython.display import Image
    display(Image(filename=str(CONFUSION_MATRIX_PATH)))
else:
    display(Markdown(
        "Confusion matrix belum tersedia. Jalankan notebook training stable sampai selesai "
        "agar file confusion matrix dibuat otomatis."
    ))

**Interpretasi:** diagonal utama menunjukkan prediksi benar. Sel di luar diagonal menunjukkan kesalahan klasifikasi. Jika dua kelas sering tertukar, pembahasan paper perlu melihat kemungkinan kemiripan gejala visual pada citra daun.

## 11. Ringkasan Narasi untuk Paper

Bagian ini merangkum temuan utama dalam bentuk kalimat yang bisa dipindahkan ke draft paper.

In [ ]:
if len(summary_df) > 0 and len(class_report_df) > 0:
    row = summary_df.iloc[0]
    accuracy_col = "test_accuracy" if "test_accuracy" in summary_df.columns else "accuracy"
    acc = float(row[accuracy_col]) * 100
    f1 = float(row["f1_macro"]) * 100
    weakest = class_report_df.sort_values("f1_score").iloc[0]
    strongest = class_report_df.sort_values("f1_score").iloc[-1]

    narrative = f"""
    Eksperimen dilakukan pada enam kelas daun padi tanpa kelas Tungro. Pembatasan ini dilakukan karena kelas Tungro memiliki jumlah data yang lebih dominan pada dataset awal. Dengan mengeluarkan Tungro, distribusi data antar kelas menjadi lebih seimbang, yaitu berada pada rentang {min_count} sampai {max_count} gambar per kelas.

    CNN Kustom Stable memperoleh accuracy sebesar {acc:.2f}% dan F1-score macro sebesar {f1:.2f}% pada test set. F1-score macro digunakan sebagai metrik penting karena setiap kelas memiliki kontribusi yang setara dalam perhitungan rata-rata. Berdasarkan evaluasi per kelas, performa terbaik terdapat pada kelas {strongest['class_name']} dengan F1-score {strongest['f1_score'] * 100:.2f}%, sedangkan kelas yang paling sulit dikenali adalah {weakest['class_name']} dengan F1-score {weakest['f1_score'] * 100:.2f}%.

    Temuan ini menunjukkan bahwa CNN Kustom mampu mempelajari sebagian pola visual penyakit daun padi, tetapi performanya perlu dibaca bersama confusion matrix karena beberapa kelas penyakit dapat memiliki gejala visual yang mirip. Keterbatasan utama eksperimen ini adalah model masih dilatih dari awal tanpa transfer learning, sehingga kualitas hasil sangat dipengaruhi oleh jumlah data, variasi gambar, dan desain arsitektur CNN.
    """
    display(Markdown(narrative))
else:
    display(Markdown(
        "Narasi hasil final belum dapat dibuat karena ringkasan metrik atau classification report belum tersedia."
    ))

## 12. Checklist Visualisasi

Checklist ini dipakai sebelum visual dimasukkan ke paper:

- Setiap grafik punya pertanyaan yang dijawab.
- Judul grafik sudah menyampaikan insight.
- Warna utama hanya dipakai untuk menyorot pesan penting.
- Tabel tidak terlalu padat.
- Metrik accuracy tidak dibaca sendirian, tetapi disandingkan dengan F1-score macro.
- Confusion matrix digunakan untuk membahas kesalahan antar kelas.
- Klaim hasil tidak berlebihan dan tetap menyebut keterbatasan model.